# A small agent team end to end

Booster LEAP, Agentic AI, topic 7. Run a team of three agents (a triage agent and two specialists) that hand work to each other with structured messages: a message validator, a context filter, a shared state, a message budget and a cycle detector, read the log of every message, change the program, and (optionally) give the triage agent a real model with your own key.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again; the experiments tell you what to try.

## Stage 1: the team with a stand-in triage model

The only **stand-in** in stage 1 is the model in the triage agent: a fake chat model from `langchain_core` that returns a prepared reply such as `order_agent:refund` (a receiver and a goal). The graph, the shared state with its reducers, the `Command` routing, the message validator, the cycle detector and the budget are real code. The order data is a small stand-in for a real order service. Run the next cell once; it installs the packages.

In [ ]:
# Installs LangGraph and LangChain (the outputs below were produced with langgraph 1.2.24, langchain 1.5.2 and langchain-core 1.8.4;
# a later release can differ in small details). Colab may ask you to restart the runtime after an install: if so, do it and run the cells again.
%pip install -q langgraph langchain

The setup cell: imports, the support desk data (orders with owners), the list of agents, the message format (`sender`, `receiver`, `type`, `task_id`, `content`) with a validator that returns a list of problems, an allow-list of the keys that may travel inside a message, a message budget for the whole team, a cycle detector (a depth-first search over the delegation graph), the shared state (a `TypedDict` whose `messages`, `edges` and `facts` lists use a reducer, so every agent *adds*), and the stand-in model. It prints nothing.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

CUSTOMER = "C2"                       # the customer who is logged in (topic 2: identity comes from the session, not from a model)
ORDERS = {"A1001": {"owner": "C1", "status": "shipped"}, "A1002": {"owner": "C2", "status": "delivered"}, "A1003": {"owner": "C1", "status": "processing"}}
AGENTS = ["triage_agent", "order_agent", "refund_agent"]
TYPES = ["task", "result", "error"]
FIELDS = ["sender", "receiver", "type", "task_id", "content"]
ALLOWED_CONTENT = ["goal", "order_id", "status"]       # the context filter: only these keys may travel inside a message (an allow-list)
BUDGET = {"max_steps": 6}                              # the most messages one request may cost, for the whole team
SETTINGS = {"refund_asks_back": False}                 # a switch for an experiment: the refund agent hands the task back to the order agent

def validate(message):
    """The message validator. Returns a list of problems; an empty list means the message is well formed."""
    missing = [f"missing field {name}" for name in FIELDS if name not in message]
    if missing:
        return missing
    problems = []
    if message["type"] not in TYPES:
        problems.append(f"unknown type {message['type']!r}")
    if message["sender"] not in AGENTS:
        problems.append(f"unknown sender {message['sender']!r}")
    if message["receiver"] not in AGENTS:
        problems.append(f"unknown receiver {message['receiver']!r}")
    if message["sender"] == message["receiver"]:
        problems.append("an agent cannot send a message to itself")
    if not isinstance(message["content"], dict):
        problems.append("content must be an object")
    return problems

def find_cycle(edges):
    """The cycle detector: a depth-first search over the delegation graph (who handed work to whom). Returns the cycle as a list of names, or None."""
    graph = {}
    for sender, receiver in edges:
        graph.setdefault(sender, []).append(receiver)
    state, path = {}, []
    def visit(node):
        state[node] = "open"
        path.append(node)
        for nxt in graph.get(node, []):
            if state.get(nxt) == "open":
                return path[path.index(nxt):] + [nxt]
            if nxt not in state:
                found = visit(nxt)
                if found:
                    return found
        path.pop()
        state[node] = "done"
        return None
    for start in list(graph):
        if start not in state:
            found = visit(start)
            if found:
                return found
    return None

class State(TypedDict):                                 # the shared state of the team
    request: str                                        # what the customer typed (may hold private data)
    order_id: str
    messages: Annotated[list, operator.add]             # the log of structured messages (reducer: every agent ADDS)
    edges: Annotated[list, operator.add]                # who delegated to whom
    facts: Annotated[list, operator.add]                # what the workers found out
    steps: int                                          # messages sent so far (the budget counter)
    answer: str
    stopped: str

class RefundView(TypedDict):                            # the only keys the refund agent receives: no request text, no other data
    order_id: str
    messages: list
    edges: list
    facts: list
    steps: int

class StandIn(GenericFakeChatModel):
    """A stand-in for the MODEL only: it returns the prepared reply and ignores its input."""

The agents. `send` is the one function every hand-over goes through: it filters the content, validates the message, checks the budget and checks the delegation graph for a cycle, and then either hands control to the receiver with `Command(goto=...)` or ends the run with a partial report. `triage`, `order_agent` and `refund_agent` are nodes that return a `Command`. The refund agent is added with `input_schema=RefundView`, so it receives only five keys of the shared state. `build` wires the graph: one edge from `START` to `triage`, and no other edges, because the agents choose where control goes. It prints nothing.

In [ ]:
def send(state, sender, receiver, kind, content, new_facts=()):
    """Every message of the team goes through here: filter, validate, check the budget, check for a cycle, then log it and hand over."""
    content = {key: value for key, value in content.items() if key in ALLOWED_CONTENT}
    message = {"sender": sender, "receiver": receiver, "type": kind, "task_id": f"T{state['steps'] + 1}", "content": content}
    problems = validate(message)
    cycle = None if problems else find_cycle(state["edges"] + [(sender, receiver)])
    if problems:
        why = "invalid message: " + "; ".join(problems)
    elif state["steps"] >= BUDGET["max_steps"]:
        why = f"message budget of {BUDGET['max_steps']} used up"
    elif cycle:
        why = "delegation cycle " + " -> ".join(cycle)
    else:
        return Command(goto=receiver, update={"messages": [message], "edges": [(sender, receiver)], "facts": list(new_facts), "steps": state["steps"] + 1})
    so_far = "; ".join(state["facts"] + list(new_facts)) or "nothing yet"
    return Command(goto=END, update={"stopped": why, "facts": list(new_facts), "answer": f"Partial report. Stopped: {why}. Found so far: {so_far}."})

def make_triage(model):
    def triage(state: State) -> Command[Literal["order_agent", "refund_agent", "__end__"]]:
        reply = model.invoke([("system", "Reply as receiver:goal, for example order_agent:status."), ("human", state["request"])]).content.strip()
        receiver, _, goal = reply.partition(":")
        return send(state, "triage_agent", receiver, "task", {"goal": goal, "order_id": state["order_id"], "raw_text": state["request"]})   # raw_text is dropped by the filter
    return triage

def order_agent(state: State) -> Command[Literal["refund_agent", "__end__"]]:
    task = state["messages"][-1]["content"]
    order = ORDERS.get(task["order_id"])
    if order is None or order["owner"] != CUSTOMER:                       # the ownership rule is ours, not the framework's (topic 2)
        return Command(goto=END, update={"answer": f"There is no order {task['order_id']} on this account.", "facts": [f"no order {task['order_id']} on this account"]})
    fact = f"{task['order_id']} is {order['status']}"
    if task["goal"] == "refund" and order["status"] == "delivered":       # hand the task over, with only the filtered content
        return send(state, "order_agent", "refund_agent", "task", {"goal": "refund", "order_id": task["order_id"], "status": order["status"]}, [fact])
    return Command(goto=END, update={"answer": f"Order {task['order_id']} is {order['status']}.", "facts": [fact]})

def refund_agent(state: RefundView) -> Command[Literal["order_agent", "__end__"]]:
    print("   refund_agent received the keys:", sorted(state))
    task = state["messages"][-1]["content"]
    if SETTINGS["refund_asks_back"]:                                      # an experiment: hand the task back to the agent that just handed it over
        return send(state, "refund_agent", "order_agent", "task", {"goal": "status", "order_id": task["order_id"]})
    return Command(goto=END, update={"answer": f"Refund for {task['order_id']} queued for approval.", "facts": [f"refund for {task['order_id']} queued"]})

def build(model):
    graph = StateGraph(State)
    graph.add_node("triage", make_triage(model))
    graph.add_node("order_agent", order_agent)
    graph.add_node("refund_agent", refund_agent, input_schema=RefundView)   # a per-node view of the shared state
    graph.add_edge(START, "triage")                                          # no other edges: the agents choose with Command
    return graph.compile()

Six runs. Each prints the log of messages (who sent what to whom), the answer, the reason for a stop if there was one, and the number of messages sent. Run 2 has a card number in the customer's text, and an assertion checks that it is nowhere in the message log. Run 4 names an agent that does not exist, run 5 makes the refund agent hand the task back (a cycle), and run 6 sets a budget of one message.

In [ ]:
def run(label, request, order_id, reply):
    print(label)
    graph = build(StandIn(messages=iter([AIMessage(content=reply)])))
    final = graph.invoke({"request": request, "order_id": order_id, "messages": [], "edges": [], "facts": [], "steps": 0, "answer": "", "stopped": ""})
    for number, message in enumerate(final["messages"], 1):
        print(f"   message {number}: {message['sender']} -> {message['receiver']} [{message['type']}] {message['task_id']} {message['content']}")
    print("   answer:", final["answer"], "| stopped:", final["stopped"] or "no", "| messages sent:", final["steps"])
    return final

run("1. a status question", "Where is my order A1002?", "A1002", "order_agent:status")
final = run("2. a refund, with a card number in the request", "Refund A1002, it arrived broken. My card is 4242 4242 4242 4242.", "A1002", "order_agent:refund")
assert "4242" not in str(final["messages"]), "private data travelled inside a message"
run("3. an order of another customer", "Refund A1003 please.", "A1003", "order_agent:refund")
run("4. a message to an agent that does not exist", "I have a billing question about A1002.", "A1002", "billing_agent:refund")
SETTINGS["refund_asks_back"] = True
run("5. the refund agent hands the task back to the order agent", "Refund A1002, it arrived broken.", "A1002", "order_agent:refund")
SETTINGS["refund_asks_back"] = False
BUDGET["max_steps"] = 1
run("6. a budget of one message", "Refund A1002, it arrived broken.", "A1002", "order_agent:refund")
BUDGET["max_steps"] = 6

1. a status question
   message 1: triage_agent -> order_agent [task] T1 {'goal': 'status', 'order_id': 'A1002'}
   answer: Order A1002 is delivered. | stopped: no | messages sent: 1
2. a refund, with a card number in the request
   refund_agent received the keys: ['edges', 'facts', 'messages', 'order_id', 'steps']
   message 1: triage_agent -> order_agent [task] T1 {'goal': 'refund', 'order_id': 'A1002'}
   message 2: order_agent -> refund_agent [task] T2 {'goal': 'refund', 'order_id': 'A1002', 'status': 'delivered'}
   answer: Refund for A1002 queued for approval. | stopped: no | messages sent: 2
3. an order of another customer
   message 1: triage_agent -> order_agent [task] T1 {'goal': 'refund', 'order_id': 'A1003'}
   answer: There is no order A1003 on this account. | stopped: no | messages sent: 1
4. a message to an agent that does not exist
   answer: Partial report. Stopped: invalid message: unknown receiver 'billing_agent'. Found so far: nothing yet. | stopped: invalid message

## Experiments (stage 1)

Change one thing at a time, predict the output, then run again (run the setup cell first when you change it, and the agents cell when you change a node).

1. Change the triage reply in run 1 to `Order_Agent:status`. What happens, and which part of the program notices?
2. Set `ALLOWED_CONTENT = ["goal"]` in the setup cell and run the refund request. What happens, and what does it tell you about a filter?
3. In run 5 set `BUDGET["max_steps"] = 2` before the run, then 6. Which guard stops the run in each case, and why?
4. Add a third agent, `policy_agent`. First reply `policy_agent:policy` with the team unchanged: what does the validator say? Then add the agent: its name in `AGENTS`, a node, `add_node`, and the `Literal` destinations of the nodes that may call it.
5. Set `CUSTOMER = "C1"` and run the refund request for A1002.
6. Make `send` check the cycle before the budget. Does any run change? Which order would you choose, and why?

## Stage 2 (optional): a real model in the triage agent

Stage 1 used a stand-in triage model. This stage gives the same team a real model with **your own key**. Read before you run:

- It sends the request text to the provider and costs a small amount of money. Each run is one short call, but you pay for what you run.
- Never write the key into a cell or share a notebook that contains it. The cell reads it from Colab's Secrets.
- A real model may reply with a sentence instead of `receiver:goal`. The validator then reports an unknown receiver and the run ends with a partial report, which is the safe result. Look at the `stopped:` line to see what happened.
- The validator, the filter, the budget, the cycle detector and the ownership rule are the same code. A stronger instruction in the triage prompt (for example, naming the three allowed receivers) can change what the model says; try it.

In [ ]:
# Optional stage 2: the stand-in model of the triage agent is replaced by a real model. This sends the request text to the provider and costs a little money.
# One-time set-up: open the key symbol (Secrets) in the left bar of Colab, add a secret named OPENAI_API_KEY and switch "Notebook access" on.
%pip install -q "langchain[openai]"
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "PUT-A-MODEL-NAME-HERE"       # a chat model of your account (see the provider's documentation)
assert MODEL != "PUT-A-MODEL-NAME-HERE", "choose a model first"
real_triage_model = init_chat_model(MODEL, model_provider="openai")

In [ ]:
def run_real(label, request, order_id):
    print(label)
    graph = build(real_triage_model)                          # only the triage model changed
    final = graph.invoke({"request": request, "order_id": order_id, "messages": [], "edges": [], "facts": [], "steps": 0, "answer": "", "stopped": ""})
    for number, message in enumerate(final["messages"], 1):
        print(f"   message {number}: {message['sender']} -> {message['receiver']} [{message['type']}] {message['content']}")
    print("   answer:", final["answer"], "| stopped:", final["stopped"] or "no")

run_real("a status question", "Where is my order A1002?", "A1002")
run_real("a refund request", "I want my money back for A1002, it arrived broken.", "A1002")

## What to look at

1. What did the real model reply in the triage step for each request? Was it `receiver:goal`, or did the validator have to stop the run?
2. Did the refund request reach the refund agent? Which agent said the last word?
3. Change the system text in `make_triage` so that the model must reply with one of the three agent names and a goal. Does the real model follow it every time? Run the same request three times.
4. Write down one case where the real model and the stand-in differ, and which part of the program protected you.